# 02  Name Candidates, Evidence, and Safeguards

This consolidated notebook replaces 03A and 04A.

It generates conservative name-enhanced candidate pairs, applies the professor-reviewed safeguards, attaches transparent pair evidence, and saves a separate v2 candidate universe. It does not accept matches, train a model, change clusters, or assign IDs.


**What this code does:** Defines inputs, versioned output paths, and limits for the conservative candidate-generation stage.


In [1]:
# Cell 1  Imports, paths, and versioned outputs

from collections import Counter
from itertools import combinations
from pathlib import Path
import re

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").is_dir():
    raise FileNotFoundError(f"Could not locate the project root from: {Path.cwd()}")

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REVIEWED_DIR = PROJECT_ROOT / "data" / "reviewed"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
REVIEW_DIR = PROJECT_ROOT / "outputs" / "review"

ENRICHED_CONTEXT_PATH = INTERIM_DIR / "persona_context_name_enriched_v1.parquet"
ORIGINAL_CANDIDATES_PATH = INTERIM_DIR / "candidate_pairs.parquet"

AUGMENTED_CANDIDATES_PATH = INTERIM_DIR / "candidate_pairs_name_enhanced_v2.parquet"
EVIDENCE_PATH = PROCESSED_DIR / "name_enhanced_candidate_pair_evidence_v2.parquet"
GENERATION_SUMMARY_PATH = REPORT_DIR / "name_enhanced_candidate_generation_summary_v2.csv"
RULE_SUMMARY_PATH = REPORT_DIR / "name_enhanced_candidate_rule_summary_v2.csv"
SAFEGUARD_AUDIT_PATH = REPORT_DIR / "name_enhanced_safeguard_rejections_v2.csv"
EVIDENCE_SUMMARY_PATH = REPORT_DIR / "name_enhanced_candidate_evidence_summary_v2.csv"
QUEUE_PATH = REVIEW_DIR / "name_enhanced_candidate_review_queue_v2.csv"
NEW_PAIR_SAMPLE_PATH = REVIEW_DIR / "name_enhanced_new_candidate_pairs_sample_v2.csv"
REVIEWED_LABELS_PATH = REVIEWED_DIR / "name_enhanced_candidate_review_labels_v2.csv"
RETAINED_REVIEW_AUDIT_PATH = REPORT_DIR / "name_enhanced_previously_confirmed_pair_retention_v2.csv"

MAX_NEW_CANDIDATE_PAIRS = 50000
MINIMUM_WITNESS_OR_GODPARENT_AGE = 12
SAME_COUPLE_CHILD_BAPTISM_GAP_YEARS = 50

for directory in [INTERIM_DIR, PROCESSED_DIR, REPORT_DIR, REVIEW_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

for required_path in [ENRICHED_CONTEXT_PATH, ORIGINAL_CANDIDATES_PATH]:
    assert required_path.exists(), f"Missing required input: {required_path}"

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Users\samav\OneDrive\Desktop\sundondo


**What this code does:** Reads enriched persona context and the original candidate pairs, then checks their required identifiers.


In [2]:
# Cell 2  Load and validate source context and original candidates

context = pd.read_parquet(ENRICHED_CONTEXT_PATH).copy()
original_candidates = pd.read_parquet(ORIGINAL_CANDIDATES_PATH).copy()

required_context_columns = {
    "persona_idno", "event_key", "event_type", "event_year", "role_standard",
    "full_name_clean", "first_name_canonical_key", "lastname_canonical_key",
    "first_name_rarity_category", "existing_gender_evidence", "event_place_linkage_key",
}
required_candidate_columns = {"persona_id_l", "persona_id_r"}

missing_context = required_context_columns - set(context.columns)
missing_candidates = required_candidate_columns - set(original_candidates.columns)
assert not missing_context, f"Enhanced context is missing: {sorted(missing_context)}"
assert not missing_candidates, f"Original candidates are missing: {sorted(missing_candidates)}"
assert not context["persona_idno"].duplicated().any(), "Context must have one row per persona."

context["persona_idno"] = context["persona_idno"].astype(str)
context["event_year_numeric"] = pd.to_numeric(context["event_year"], errors="coerce")

print("Enriched context rows:", len(context))
print("Original candidate pairs:", len(original_candidates))


Enriched context rows: 47072
Original candidate pairs: 180886


**What this code does:** Sets the reviewed name, gender, chronology, role, and family safeguards that candidate pairs must satisfy.


In [3]:
# Cell 3  Define matching rules and professor-reviewed safeguards

def nonempty(value):
    return pd.notna(value) and str(value).strip().lower() not in {"", "nan", "none"}

def normalized_text(value):
    return re.sub(r"\s+", " ", str(value).strip().lower())

def pair_key(left_id, right_id):
    return "||".join(sorted([str(left_id), str(right_id)]))

def safeguard_role(row):
    values = [row.get(column, "") for column in ["role_standard", "persona_type", "raw_role_clean", "role"]]
    text = normalized_text(" ".join(str(value) for value in values if nonempty(value)))
    if any(token in text for token in ["godparent", "godfather", "godmother", "padrino", "madrina"]):
        return "godparent"
    if any(token in text for token in ["witness", "testigo"]):
        return "witness"
    if "baptized" in text or "bautizado" in text or "bautizada" in text:
        return "baptized"
    if "mother" in text or "madre" in text:
        return "mother"
    if "father" in text or "padre" in text:
        return "father"
    return normalized_text(row.get("role_standard", ""))

def pair_safeguard_reason(left, right):
    left_year, right_year = left["event_year_numeric"], right["event_year_numeric"]
    if pd.isna(left_year) or pd.isna(right_year):
        return None

    left_role, right_role = safeguard_role(left), safeguard_role(right)
    left_event, right_event = normalized_text(left["event_type"]), normalized_text(right["event_type"])

    # Professor rule 1: a person cannot be a witness/godparent before age 12
    # or before their own baptism.
    left_is_own_baptism = left_event == "baptism" and left_role == "baptized"
    right_is_own_baptism = right_event == "baptism" and right_role == "baptized"
    left_is_service_role = left_role in {"witness", "godparent"}
    right_is_service_role = right_role in {"witness", "godparent"}

    if left_is_own_baptism and right_is_service_role:
        if right_year - left_year < MINIMUM_WITNESS_OR_GODPARENT_AGE:
            return "witness_or_godparent_under_12_or_before_baptism"
    if right_is_own_baptism and left_is_service_role:
        if left_year - right_year < MINIMUM_WITNESS_OR_GODPARENT_AGE:
            return "witness_or_godparent_under_12_or_before_baptism"

    # Professor rule 2: the same named parent and spouse cannot credibly have
    # child baptisms fifty or more years apart.
    same_parent_role = (
        left_event == "baptism" and right_event == "baptism"
        and left_role == right_role and left_role in {"mother", "father"}
    )
    same_spouse = (
        nonempty(left.get("spouse_name_clean_ctx", ""))
        and left.get("spouse_name_clean_ctx", "") == right.get("spouse_name_clean_ctx", "")
    )
    if same_parent_role and same_spouse and abs(left_year - right_year) >= SAME_COUPLE_CHILD_BAPTISM_GAP_YEARS:
        return "same_named_couple_child_baptisms_50_or_more_years_apart"

    return None

def pair_is_compatible(left, right):
    if left["persona_idno"] == right["persona_idno"]:
        return False, "same_persona"
    if (
        nonempty(left["event_key"]) and left["event_key"] == right["event_key"]
        and left["role_standard"] == right["role_standard"]
    ):
        return False, "same_event_same_role"

    left_gender = normalized_text(left["existing_gender_evidence"])
    right_gender = normalized_text(right["existing_gender_evidence"])
    if {left_gender, right_gender} == {"male", "female"}:
        return False, "gender_conflict"

    left_year, right_year = left["event_year_numeric"], right["event_year_numeric"]
    if pd.notna(left_year) and pd.notna(right_year):
        year_gap = abs(left_year - right_year)
        if year_gap > 80:
            return False, "event_year_gap_over_80"
        if left["role_standard"] == "baptized" and right["role_standard"] == "baptized" and year_gap > 1:
            return False, "two_baptized_roles_more_than_one_year_apart"

    safeguard_reason = pair_safeguard_reason(left, right)
    if safeguard_reason:
        return False, safeguard_reason
    return True, "compatible"

context["canonical_full_name_key"] = (
    context["first_name_canonical_key"].fillna("").str.strip()
    + " | " + context["lastname_canonical_key"].fillna("").str.strip()
)
valid_name = (
    context["first_name_canonical_key"].map(nonempty)
    & context["lastname_canonical_key"].map(nonempty)
)
candidate_rules = [{
    "rule_name": "canonical_full_name_rare",
    "block_columns": ["canonical_full_name_key"],
    "maximum_block_size": 12,
    "strength": 1,
    "mask": valid_name & context["first_name_rarity_category"].isin(["Rare", "Very rare"]),
}]
if "parent_pair_clean" in context.columns:
    candidate_rules.append({
        "rule_name": "canonical_full_name_plus_parent_pair",
        "block_columns": ["canonical_full_name_key", "parent_pair_clean"],
        "maximum_block_size": 12, "strength": 2,
        "mask": valid_name & context["parent_pair_clean"].map(nonempty),
    })
if "spouse_name_clean_ctx" in context.columns:
    candidate_rules.append({
        "rule_name": "canonical_full_name_plus_spouse",
        "block_columns": ["canonical_full_name_key", "spouse_name_clean_ctx"],
        "maximum_block_size": 12, "strength": 2,
        "mask": valid_name & context["spouse_name_clean_ctx"].map(nonempty),
    })

# Marriage godparents were previously unmapped and therefore absent from the
# candidate pool.  This is a candidate-only rule: exact canonical full name +
# normalized event place, with at least one marriage godparent in the pair.
# It cannot itself accept a match; the normal safeguards and later scoring still apply.
valid_name_place = valid_name & context["event_place_linkage_key"].map(nonempty)
candidate_rules.append({
    "rule_name": "marriage_godparent_canonical_full_name_plus_place",
    "block_columns": ["canonical_full_name_key", "event_place_linkage_key"],
    "maximum_block_size": 12,
    "strength": 1,
    "mask": valid_name_place,
    "requires_any_role": "godparent",
})

print("Enabled rules:", [rule["rule_name"] for rule in candidate_rules])


Enabled rules: ['canonical_full_name_rare', 'canonical_full_name_plus_parent_pair', 'canonical_full_name_plus_spouse', 'marriage_godparent_canonical_full_name_plus_place']


**What this code does:** Creates only plausible additional pairs and records why any candidate was rejected or held.


In [4]:
# Cell 4  Generate conservative candidates and audit every rejection reason

generated_rows = []
rule_audit_rows = []
rejection_counts = Counter()
context_rows = context.to_dict(orient="index")

for rule in candidate_rules:
    rule_context = context.loc[rule["mask"]].copy()
    required_role = rule.get("requires_any_role")
    # For service-role rules, scan only name/place blocks which actually contain
    # the requested role. This avoids iterating through thousands of irrelevant
    # singleton blocks while preserving the exact candidate definition.
    if required_role:
        role_blocks = rule_context.loc[
            rule_context["role_standard"].eq(required_role), rule["block_columns"]
        ].drop_duplicates()
        if role_blocks.empty:
            rule_context = rule_context.iloc[0:0].copy()
        else:
            all_block_index = pd.MultiIndex.from_frame(rule_context[rule["block_columns"]])
            role_block_index = pd.MultiIndex.from_frame(role_blocks)
            rule_context = rule_context.loc[all_block_index.isin(role_block_index)].copy()
    raw_pairs = accepted_pairs = oversized_blocks = 0

    for _, group in rule_context.groupby(rule["block_columns"], dropna=False, sort=False):
        if len(group) < 2:
            continue
        if len(group) > rule["maximum_block_size"]:
            oversized_blocks += 1
            continue

        for left_index, right_index in combinations(list(group.index), 2):
            left, right = context_rows[left_index], context_rows[right_index]
            if required_role and required_role not in {left["role_standard"], right["role_standard"]}:
                continue
            raw_pairs += 1
            compatible, reason = pair_is_compatible(left, right)
            if not compatible:
                rejection_counts[reason] += 1
                continue

            accepted_pairs += 1
            generated_rows.append({
                "pair_key": pair_key(left["persona_idno"], right["persona_idno"]),
                "persona_id_l": min(left["persona_idno"], right["persona_idno"]),
                "persona_id_r": max(left["persona_idno"], right["persona_idno"]),
                "blocking_rule": rule["rule_name"],
                "blocking_strength": rule["strength"],
            })

    rule_audit_rows.append({
        "rule_name": rule["rule_name"],
        "eligible_personas": len(rule_context),
        "raw_pair_combinations": raw_pairs,
        "compatible_pair_combinations": accepted_pairs,
        "oversized_blocks_skipped": oversized_blocks,
    })

generated_pairs = pd.DataFrame(
    generated_rows,
    columns=["pair_key", "persona_id_l", "persona_id_r", "blocking_rule", "blocking_strength"],
)
rule_summary = pd.DataFrame(rule_audit_rows)
safeguard_audit = pd.DataFrame(
    [{"rejection_reason": reason, "rejected_pair_combinations": count} for reason, count in sorted(rejection_counts.items())],
    columns=["rejection_reason", "rejected_pair_combinations"],
)

display(rule_summary)
display(safeguard_audit)


,rule_name,eligible_personas,raw_pair_combinations,compatible_pair_combinations,oversized_blocks_skipped
0,canonical_full_name_rare,950,21,21,0
1,canonical_full_name_plus_parent_pair,10129,267,253,0
2,canonical_full_name_plus_spouse,16780,5519,5516,0
3,marriage_godparent_canonical_full_name_plus_place,10976,6953,6701,180


,rejection_reason,rejected_pair_combinations
0,event_year_gap_over_80,118
1,same_event_same_role,1
2,same_named_couple_child_baptisms_50_or_more_ye...,3
3,two_baptized_roles_more_than_one_year_apart,14
4,witness_or_godparent_under_12_or_before_baptism,133


**What this code does:** Writes the new v2 candidate universe separately and verifies that the original candidate pairs remain intact.


In [5]:
# Cell 5  Save a separate v2 augmented candidate universe

original_pair_keys = set(
    original_candidates.apply(lambda row: pair_key(row["persona_id_l"], row["persona_id_r"]), axis=1)
)

if generated_pairs.empty:
    new_candidates = generated_pairs.copy()
else:
    new_candidates = (
        generated_pairs.groupby(["pair_key", "persona_id_l", "persona_id_r"], as_index=False)
        .agg(
            blocking_rules=("blocking_rule", lambda values: " || ".join(sorted(set(values)))),
            blocking_rule_count=("blocking_rule", "nunique"),
            blocking_strength_max=("blocking_strength", "max"),
            blocking_strength_sum=("blocking_strength", "sum"),
        )
    )
    new_candidates = new_candidates[~new_candidates["pair_key"].isin(original_pair_keys)].copy()

# Preserve completed, previously confirmed review pairs even if a later
# canonicalization changes the blocking condition that originally surfaced them.
retained_review_pairs = pd.DataFrame(columns=["pair_key", "persona_id_l", "persona_id_r"])
if REVIEWED_LABELS_PATH.exists():
    completed_labels = pd.read_csv(REVIEWED_LABELS_PATH, dtype=str).fillna("")
    required_label_columns = {"pair_key", "persona_id_l", "persona_id_r", "reviewer_decision"}
    missing_label_columns = required_label_columns - set(completed_labels.columns)
    assert not missing_label_columns, f"Completed labels are missing: {sorted(missing_label_columns)}"
    retained_review_pairs = completed_labels.loc[
        completed_labels["reviewer_decision"].str.strip().str.lower().eq("confirmed_match"),
        ["pair_key", "persona_id_l", "persona_id_r"],
    ].drop_duplicates("pair_key")
    retained_review_pairs = retained_review_pairs[~retained_review_pairs["pair_key"].isin(original_pair_keys)]
    retained_review_pairs = retained_review_pairs[~retained_review_pairs["pair_key"].isin(new_candidates["pair_key"])]

    if not retained_review_pairs.empty:
        retained_review_pairs["blocking_rules"] = "previously_reviewed_confirmed_retained"
        retained_review_pairs["blocking_rule_count"] = 0
        retained_review_pairs["blocking_strength_max"] = 0
        retained_review_pairs["blocking_strength_sum"] = 0
        retained_review_pairs["found_by_primary"] = False
        retained_review_pairs["found_by_rescue"] = True
        retained_review_pairs["candidate_stage"] = "previously_reviewed_confirmed_retained"
        new_candidates = pd.concat([new_candidates, retained_review_pairs], ignore_index=True, sort=False)

new_candidates["found_by_primary"] = False
new_candidates["found_by_rescue"] = True
new_candidates["candidate_stage"] = new_candidates.get(
    "candidate_stage", pd.Series(pd.NA, index=new_candidates.index)
).fillna("name_enhancement_rescue_v2")

if len(new_candidates) > MAX_NEW_CANDIDATE_PAIRS:
    raise RuntimeError(f"Generated {len(new_candidates):,} pairs, exceeding the safety ceiling.")

for column in original_candidates.columns:
    if column not in new_candidates.columns:
        new_candidates[column] = pd.NA
new_candidates = new_candidates[original_candidates.columns]
augmented_candidates = pd.concat([original_candidates, new_candidates], ignore_index=True)

assert len(augmented_candidates) == len(original_candidates) + len(new_candidates)
assert len(new_candidates) <= MAX_NEW_CANDIDATE_PAIRS

generation_summary = pd.DataFrame([
    {"metric": "original_candidate_pairs", "value": len(original_candidates)},
    {"metric": "new_name_enhancement_pairs_v2", "value": len(new_candidates)},
    {"metric": "augmented_candidate_pairs_v2", "value": len(augmented_candidates)},
    {"metric": "original_pairs_preserved", "value": 1},
    {"metric": "previously_confirmed_pairs_added_back", "value": len(retained_review_pairs)},
    {"metric": "previously_confirmed_pair_retention_enabled", "value": int(REVIEWED_LABELS_PATH.exists())},
    {"metric": "new_pair_safety_ceiling", "value": MAX_NEW_CANDIDATE_PAIRS},
])
retention_audit = retained_review_pairs.copy()
retention_audit["retention_reason"] = "previously_confirmed_pair_not_regenerated_after_name_canonicalization"

augmented_candidates.to_parquet(AUGMENTED_CANDIDATES_PATH, index=False)
generation_summary.to_csv(GENERATION_SUMMARY_PATH, index=False)
rule_summary.to_csv(RULE_SUMMARY_PATH, index=False)
safeguard_audit.to_csv(SAFEGUARD_AUDIT_PATH, index=False)
retention_audit.to_csv(RETAINED_REVIEW_AUDIT_PATH, index=False)

review_fields = [column for column in [
    "persona_id_l", "persona_id_r", "blocking_rules", "blocking_rule_count",
    "blocking_strength_max", "candidate_stage",
] if column in new_candidates.columns]
new_candidates[review_fields].head(500).to_csv(NEW_PAIR_SAMPLE_PATH, index=False)

display(generation_summary)
print(f"Saved v2 augmented candidates: {AUGMENTED_CANDIDATES_PATH}")
print(f"Saved safeguard audit: {SAFEGUARD_AUDIT_PATH}")


,metric,value
0,original_candidate_pairs,180886
1,new_name_enhancement_pairs_v2,8093
2,augmented_candidate_pairs_v2,188979
3,original_pairs_preserved,1
4,previously_confirmed_pairs_added_back,2
5,previously_confirmed_pair_retention_enabled,1
6,new_pair_safety_ceiling,50000


Saved v2 augmented candidates: C:\Users\samav\OneDrive\Desktop\sundondo\data\interim\candidate_pairs_name_enhanced_v2.parquet
Saved safeguard audit: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\reports\name_enhanced_safeguard_rejections_v2.csv


## Pair-level review evidence

The following cells attach the enriched persona context to only the newly added v2 pairs. The evidence is for review and audit; it is not an automatic acceptance decision.


**What this code does:** Joins each newly added pair to its two personas so reviewers can inspect names, relatives, dates, and roles.


In [6]:
# Cell 6  Identify v2 pairs and attach persona evidence

augmented_candidates = pd.read_parquet(AUGMENTED_CANDIDATES_PATH).copy()
for frame in [original_candidates, augmented_candidates]:
    frame["pair_key"] = frame.apply(lambda row: pair_key(row["persona_id_l"], row["persona_id_r"]), axis=1)

new_pairs = augmented_candidates[
    ~augmented_candidates["pair_key"].isin(original_pair_keys)
].copy()
assert len(new_pairs) == len(new_candidates), "Saved v2 candidate count changed unexpectedly."

required_evidence_columns = {
    "persona_idno", "full_name_clean", "event_key", "event_type", "event_year",
    "role_standard", "parent_pair_clean", "spouse_name_clean_ctx",
    "first_name_canonical_key", "lastname_canonical_key", "first_name_frequency",
    "first_name_rarity_category", "existing_gender_evidence", "event_place_linkage_key",
    "first_name_gender_evidence", "gender_evidence_status",
}
missing_evidence_columns = required_evidence_columns - set(context.columns)
assert not missing_evidence_columns, f"Context is missing evidence columns: {sorted(missing_evidence_columns)}"

context_columns = list(required_evidence_columns - {"persona_idno"})
inherited_columns = [
    f"{column}_{side}" for column in context_columns for side in ["l", "r"]
    if f"{column}_{side}" in new_pairs.columns
]
pair_input = new_pairs.drop(columns=inherited_columns).copy()

left_context = context[["persona_idno"] + context_columns].rename(columns={column: f"{column}_l" for column in context_columns})
right_context = context[["persona_idno"] + context_columns].rename(columns={column: f"{column}_r" for column in context_columns})

evidence = pair_input.merge(left_context, left_on="persona_id_l", right_on="persona_idno", how="left", validate="m:1").drop(columns="persona_idno")
evidence = evidence.merge(right_context, left_on="persona_id_r", right_on="persona_idno", how="left", validate="m:1").drop(columns="persona_idno")

assert len(evidence) == len(new_pairs)
assert evidence["full_name_clean_l"].notna().all() and evidence["full_name_clean_r"].notna().all()
print("New v2 candidate pairs for review:", len(evidence))


New v2 candidate pairs for review: 8093


**What this code does:** Creates transparent evidence fields and review flags. This scores evidence but does not approve a match.


In [7]:
# Cell 7  Calculate transparent review evidence

def exact_nonempty(left, right):
    return left.notna() & right.notna() & left.astype(str).str.strip().ne("") & right.astype(str).str.strip().eq(right.astype(str).str.strip())

evidence["canonical_first_name_exact"] = exact_nonempty(evidence["first_name_canonical_key_l"], evidence["first_name_canonical_key_r"])
evidence["canonical_lastname_exact"] = exact_nonempty(evidence["lastname_canonical_key_l"], evidence["lastname_canonical_key_r"])
evidence["canonical_full_name_exact"] = evidence["canonical_first_name_exact"] & evidence["canonical_lastname_exact"]
evidence["parent_pair_exact"] = exact_nonempty(evidence["parent_pair_clean_l"], evidence["parent_pair_clean_r"])
evidence["spouse_name_exact"] = exact_nonempty(evidence["spouse_name_clean_ctx_l"], evidence["spouse_name_clean_ctx_r"])
# Place is supporting evidence only: it can raise review priority but never admits or merges a pair on its own.
place_key_l = evidence["event_place_linkage_key_l"].fillna("").astype(str).str.strip()
place_key_r = evidence["event_place_linkage_key_r"].fillna("").astype(str).str.strip()
evidence["place_linkage_exact"] = place_key_l.ne("") & place_key_l.eq(place_key_r)
assert evidence["place_linkage_exact"].eq(place_key_l.ne("") & place_key_l.eq(place_key_r)).all(), 'Place agreement must exactly match the two linkage keys.'
evidence["event_year_gap"] = (pd.to_numeric(evidence["event_year_l"], errors="coerce") - pd.to_numeric(evidence["event_year_r"], errors="coerce")).abs()
evidence["rare_first_name"] = evidence["first_name_rarity_category_l"].isin(["Rare", "Very rare"]) & evidence["first_name_rarity_category_r"].isin(["Rare", "Very rare"])
evidence["gender_evidence_conflict"] = (
    evidence["existing_gender_evidence_l"].isin(["male", "female"])
    & evidence["existing_gender_evidence_r"].isin(["male", "female"])
    & evidence["existing_gender_evidence_l"].ne(evidence["existing_gender_evidence_r"])
)
evidence["same_event_same_role"] = evidence["event_key_l"].eq(evidence["event_key_r"]) & evidence["role_standard_l"].eq(evidence["role_standard_r"])
evidence["chronology_review_flag"] = evidence["event_year_gap"].gt(80)
evidence["professor_safeguards_applied"] = True

# The safeguards were applied before a pair was admitted, so no admitted v2 pair
# may retain either prohibited pattern.
assert not evidence["chronology_review_flag"].any(), "A v2 pair exceeds the chronological ceiling."
assert not evidence["same_event_same_role"].any(), "A v2 pair has the same event and role."

evidence["review_support_score"] = (
    2 * evidence["canonical_full_name_exact"].astype(int)
    + 2 * evidence["parent_pair_exact"].astype(int)
    + 2 * evidence["spouse_name_exact"].astype(int)
    + evidence["rare_first_name"].astype(int)
    + evidence["place_linkage_exact"].astype(int)
    - 4 * evidence["gender_evidence_conflict"].astype(int)
)
evidence["review_priority"] = pd.cut(
    evidence["review_support_score"],
    bins=[-999, 2, 4, 999],
    labels=["low_context", "medium_context", "high_context"],
    include_lowest=True,
)
evidence["review_decision"] = "unreviewed"
evidence["review_notes"] = ""


**What this code does:** Saves candidate evidence, the review queue, and validation summaries for the next notebook.


In [8]:
# Cell 8  Save v2 evidence, review queue, and validation report

evidence_summary = pd.DataFrame([
    {"metric": "new_candidate_pairs_v2", "value": len(evidence)},
    {"metric": "canonical_full_name_exact", "value": int(evidence["canonical_full_name_exact"].sum())},
    {"metric": "parent_pair_exact", "value": int(evidence["parent_pair_exact"].sum())},
    {"metric": "spouse_name_exact", "value": int(evidence["spouse_name_exact"].sum())},
    {"metric": "rare_first_name_pairs", "value": int(evidence["rare_first_name"].sum())},
    {"metric": "place_linkage_exact_pairs", "value": int(evidence["place_linkage_exact"].sum())},
    {"metric": "gender_evidence_conflicts", "value": int(evidence["gender_evidence_conflict"].sum())},
    {"metric": "chronology_review_flags", "value": int(evidence["chronology_review_flag"].sum())},
    {"metric": "same_event_same_role_flags", "value": int(evidence["same_event_same_role"].sum())},
    {"metric": "professor_safeguards_applied", "value": 1},
])

queue_columns = [
    "pair_key", "persona_id_l", "persona_id_r", "blocking_rules",
    "full_name_clean_l", "full_name_clean_r", "event_key_l", "event_key_r",
    "event_type_l", "event_type_r", "event_year_l", "event_year_r",
    "role_standard_l", "role_standard_r", "parent_pair_clean_l", "parent_pair_clean_r",
    "spouse_name_clean_ctx_l", "spouse_name_clean_ctx_r", "event_place_linkage_key_l", "event_place_linkage_key_r",
    "canonical_full_name_exact", "parent_pair_exact", "spouse_name_exact", "place_linkage_exact",
    "rare_first_name", "event_year_gap", "gender_evidence_conflict",
    "chronology_review_flag", "professor_safeguards_applied",
    "review_support_score", "review_priority", "review_decision", "review_notes",
]
queue = evidence[queue_columns].sort_values(
    ["review_support_score", "event_year_gap"], ascending=[False, True], na_position="last"
).reset_index(drop=True)

evidence.to_parquet(EVIDENCE_PATH, index=False)
evidence_summary.to_csv(EVIDENCE_SUMMARY_PATH, index=False)
queue.to_csv(QUEUE_PATH, index=False)

display(evidence_summary)
display(queue.head(30))
print(f"Saved v2 pair evidence: {EVIDENCE_PATH}")
print(f"Saved v2 review queue: {QUEUE_PATH}")


,metric,value
0,new_candidate_pairs_v2,8093
1,canonical_full_name_exact,8093
2,parent_pair_exact,19
3,spouse_name_exact,2777
4,rare_first_name_pairs,4
5,place_linkage_exact_pairs,7407
6,gender_evidence_conflicts,0
7,chronology_review_flags,0
8,same_event_same_role_flags,0
9,professor_safeguards_applied,1


,pair_key,persona_id_l,persona_id_r,blocking_rules,full_name_clean_l,full_name_clean_r,event_key_l,event_key_r,event_type_l,event_type_r,...,place_linkage_exact,rare_first_name,event_year_gap,gender_evidence_conflict,chronology_review_flag,professor_safeguards_applied,review_support_score,review_priority,review_decision,review_notes
0,persona-40094||persona-40459,persona-40094,persona-40459,canonical_full_name_plus_spouse,juana ccoillo,juana ccoyllo,APAucará-LM-L003_M328,APAucará-LM-L003_M369,marriage,marriage,...,True,False,1,False,False,True,7,high_context,unreviewed,
1,persona-37452||persona-40847,persona-37452,persona-40847,canonical_full_name_plus_parent_pair,jervacia mejia,gervacia mejia,APAucará-LM-L003_M070,APAucará-LM-L003_M415,marriage,marriage,...,True,False,14,False,False,True,7,high_context,unreviewed,
2,persona-34958||persona-46972,persona-34958,persona-46972,canonical_full_name_plus_spouse,balentin chalco,valentin chalco,APAucará-LM-L002_M151,APAucará-LE-L003_E817,marriage,burial,...,True,False,53,False,False,True,7,high_context,unreviewed,
3,persona-40233||persona-40319,persona-40233,persona-40319,canonical_full_name_rare || marriage_godparent...,arturo vargas,arturo vargas,APAucará-LM-L003_M342,APAucará-LM-L003_M351,marriage,marriage,...,True,True,1,False,False,True,6,high_context,unreviewed,
4,persona-27880||persona-7245,persona-27880,persona-7245,canonical_full_name_plus_parent_pair,eusevio flores,eusebio flores,APAucará-LM-L001_M326,APAucará-LB-L001_B1857,marriage,baptism,...,False,False,17,False,False,True,6,high_context,unreviewed,
5,persona-27918||persona-7245,persona-27918,persona-7245,canonical_full_name_plus_parent_pair,eusevio flores,eusebio flores,APAucará-LM-L001_M330,APAucará-LB-L001_B1857,marriage,baptism,...,False,False,17,False,False,True,6,high_context,unreviewed,
6,persona-10047||persona-28569,persona-10047,persona-28569,marriage_godparent_canonical_full_name_plus_place,santos guallpatuiro,santos huallpatuiro,APAucará-LB-L002_B00673,APAucará-LM-L001_M393,baptism,marriage,...,True,False,0,False,False,True,5,high_context,unreviewed,
7,persona-10110||persona-10254,persona-10110,persona-10254,canonical_full_name_plus_spouse,andrea huauia,andrea huauya,APAucará-LB-L002_B00691,APAucará-LB-L002_B00727,baptism,baptism,...,True,False,0,False,False,True,5,high_context,unreviewed,
8,persona-11241||persona-39936,persona-11241,persona-39936,marriage_godparent_canonical_full_name_plus_place,oseas bendezu,oseas bendezu,APAucará-LB-L006_B00226,APAucará-LM-L003_M313,baptism,marriage,...,True,False,0,False,False,True,5,high_context,unreviewed,
9,persona-11242||persona-39937,persona-11242,persona-39937,marriage_godparent_canonical_full_name_plus_place,tereza rodriguez,teresa rodrigues,APAucará-LB-L006_B00226,APAucará-LM-L003_M313,baptism,marriage,...,True,False,0,False,False,True,5,high_context,unreviewed,


Saved v2 pair evidence: C:\Users\samav\OneDrive\Desktop\sundondo\data\processed\name_enhanced_candidate_pair_evidence_v2.parquet
Saved v2 review queue: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\review\name_enhanced_candidate_review_queue_v2.csv
